# CS412: Bisaya Speech Affect Recognition - Machine Learning Experiments
**Course:** CS412 – Affective Computing Research Mini-Project  
**Cohort:** Group 7 (4th-Year Computer Science)  
**Deliverable:** 10-Minute Live Technical Checking – Part C: Machine Learning and Results  

---

## 1. Experimental Overview & Setup
This notebook implements the Speech Emotion Recognition (SER) modeling and benchmarking pipeline using **108 acoustic & prosodic speech features** across **40 spontaneous Bisaya post-class utterances**.

### Pipeline Strip:
`108 features → median imputer → StandardScaler → classifier → stratified 5-fold CV (+ LOOCV)`

### Key Hygiene Controls:
* **Leakage Control:** Imputer and Scaler are fit strictly inside each training fold; ground-truth self-reports and context are excluded from predictors.
* **Random Seed:** 42 across all splits and algorithms.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

sys.path.append(os.path.abspath('.'))
from src.multimodal_dataset import predictor_columns
print('[OK] Environment configured and libraries loaded!')

## 2. Dataset Loading & Feature Space Verification

In [ ]:
df = pd.read_csv('datasets/multimodal/cleaned_multimodal_dataset.csv')
audio_features = [
    c for c in predictor_columns(df, modality='audio') 
    if df[c].notna().any() and df[c].nunique(dropna=True) > 1
]

print(f'Total Participant Observations: {len(df)}')
print(f'Total Acoustic Predictors: {len(audio_features)}')
print(f'Valence Class Distribution: {dict(df["valence_group"].value_counts())}')
print(f'Arousal Class Distribution: {dict(df["arousal_group"].value_counts())}')

## 3. Model Architecture & Hyperparameters Table
| Model | Scikit-Learn Class | Key Parameters |
|---|---|---|
| Majority Baseline | `DummyClassifier` | `strategy='most_frequent'` |
| Logistic Regression | `LogisticRegression` | `penalty='l2', C=0.1, max_iter=1000, random_state=42` |
| Linear SVM | `SVC(kernel='linear')` | `kernel='linear', C=0.1, random_state=42` |
| RBF SVM | `SVC(kernel='rbf')` | `kernel='rbf', C=1.0, gamma='scale', random_state=42` |
| Random Forest | `RandomForestClassifier` | `n_estimators=100, max_depth=3, random_state=42` |

In [ ]:
X = df[audio_features].values
y_val = df['valence_group'].values
y_aro = df['arousal_group'].values

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

models = {
    'Majority Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(penalty='l2', C=0.1, max_iter=1000, random_state=42),
    'Linear SVM': SVC(kernel='linear', C=0.1, random_state=42),
    'RBF SVM': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
}

## 4. Stratified 5-Fold Cross-Validation Evaluation

In [ ]:
summary_rows = []
val_preds, aro_preds = {}, {}
val_stats, aro_stats = {}, {}

for task_name, y_target, preds_dict, stats_dict in [
    ('Valence', y_val, val_preds, val_stats),
    ('Arousal', y_aro, aro_preds, aro_stats)
]:
    for m_name, clf in models.items():
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', clf)
        ])
        cv_res = cross_validate(pipe, X, y_target, cv=cv5, scoring=scoring)
        preds_dict[m_name] = cross_val_predict(pipe, X, y_target, cv=cv5)

        acc_m = cv_res['test_accuracy'].mean()
        acc_s = cv_res['test_accuracy'].std()
        f1_m = cv_res['test_f1_macro'].mean()
        f1_s = cv_res['test_f1_macro'].std()
        prec_m = cv_res['test_precision_macro'].mean()
        prec_s = cv_res['test_precision_macro'].std()
        rec_m = cv_res['test_recall_macro'].mean()
        rec_s = cv_res['test_recall_macro'].std()

        stats_dict[m_name] = {'acc_mean': acc_m, 'acc_std': acc_s, 'f1_mean': f1_m, 'f1_std': f1_s}
        summary_rows.append({
            'Task': task_name, 'Model': m_name,
            'Accuracy': f'{acc_m*100:.1f}% ± {acc_s*100:.1f}%',
            'Macro Precision': f'{prec_m:.3f} ± {prec_s:.3f}',
            'Macro Recall': f'{rec_m:.3f} ± {rec_s:.3f}',
            'Macro F1': f'{f1_m:.3f} ± {f1_s:.3f}'
        })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

## 5. Figure 11: 2x2 Out-of-Fold Confusion Matrices Grid
* **Top Row (Valence):** Random Forest (Best: 32.5%) vs. Logistic Regression (30.0%)
* **Bottom Row (Arousal):** Linear SVM (Best: 47.5%) vs. Random Forest (45.0%)
* Cell annotations show raw counts with row-normalized recall percentages.

In [ ]:
fig_cm, axes_cm = plt.subplots(2, 2, figsize=(13, 11))
cm_configs = [
    (0, 0, 'Valence', 'Random Forest', val_preds['Random Forest'], y_val, ['Negative', 'Neutral', 'Positive'], 'Blues', val_stats['Random Forest']['acc_mean']),
    (0, 1, 'Valence', 'Logistic Regression', val_preds['Logistic Regression'], y_val, ['Negative', 'Neutral', 'Positive'], 'Blues', val_stats['Logistic Regression']['acc_mean']),
    (1, 0, 'Arousal', 'Linear SVM', aro_preds['Linear SVM'], y_aro, ['Low', 'Moderate', 'High'], 'Oranges', aro_stats['Linear SVM']['acc_mean']),
    (1, 1, 'Arousal', 'Random Forest', aro_preds['Random Forest'], y_aro, ['Low', 'Moderate', 'High'], 'Oranges', aro_stats['Random Forest']['acc_mean']),
]

for r, c, task, m_name, y_p, y_true, labels, cmap, acc in cm_configs:
    ax = axes_cm[r, c]
    cm_counts = confusion_matrix(y_true, y_p, labels=labels)
    row_sums = cm_counts.sum(axis=1)[:, np.newaxis]
    cm_norm = np.divide(cm_counts.astype(float), row_sums, out=np.zeros_like(cm_counts, dtype=float), where=row_sums != 0)
    annot_matrix = np.empty_like(cm_counts, dtype=object)
    for i in range(cm_counts.shape[0]):
        for j in range(cm_counts.shape[1]):
            annot_matrix[i, j] = f'{cm_counts[i, j]}\n({cm_norm[i, j]*100:.0f}%)'
    sns.heatmap(cm_norm, annot=annot_matrix, fmt='', cmap=cmap, vmin=0.0, vmax=1.0, cbar=(c == 1), cbar_kws={'label': 'Row-Normalized Recall'} if c == 1 else None, xticklabels=labels, yticklabels=labels, ax=ax, linewidths=1, linecolor='white', annot_kws={'fontsize': 11, 'fontweight': 'bold'})
    badge = ' [BEST MODEL]' if (m_name == 'Random Forest' and task == 'Valence') or (m_name == 'Linear SVM' and task == 'Arousal') else ' [COMPARISON]'
    ax.set_title(f'{task}: {m_name} — Acc: {acc*100:.1f}%{badge}', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted Class', fontsize=10, fontweight='bold')
    ax.set_ylabel('True Ground Truth', fontsize=10, fontweight='bold')

fig_cm.suptitle('Figure 11: Out-of-Fold Confusion Matrices (Stratified 5-Fold Cross-Validation)', fontsize=14, fontweight='bold', y=0.98)
fig_cm.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 6. Figure 12: Model Performance Comparison Bar Chart
Side-by-side comparison across all 5 models for Valence and Arousal with error bars ($\pm 1\text{ SD}$) and chance/baseline thresholds.

In [ ]:
fig_bar, axes_bar = plt.subplots(1, 2, figsize=(15, 6))
model_order = ['Majority Baseline', 'Logistic Regression', 'Linear SVM', 'RBF SVM', 'Random Forest']
x = np.arange(len(model_order))
w = 0.35

for idx, (task, pal_acc, pal_f1, base_val) in enumerate([('Valence', '#457B9D', '#2A9D8F', 35.0), ('Arousal', '#E76F51', '#F4A261', 40.0)]):
    ax = axes_bar[idx]
    acc_means = [val_stats[m]['acc_mean']*100 if task=='Valence' else aro_stats[m]['acc_mean']*100 for m in model_order]
    acc_stds = [val_stats[m]['acc_std']*100 if task=='Valence' else aro_stats[m]['acc_std']*100 for m in model_order]
    f1_means = [val_stats[m]['f1_mean']*100 if task=='Valence' else aro_stats[m]['f1_mean']*100 for m in model_order]
    f1_stds = [val_stats[m]['f1_std']*100 if task=='Valence' else aro_stats[m]['f1_std']*100 for m in model_order]

    ax.bar(x - w/2, acc_means, w, yerr=acc_stds, capsize=4, label='Accuracy', color=pal_acc, edgecolor='black', alpha=0.9)
    ax.bar(x + w/2, f1_means, w, yerr=f1_stds, capsize=4, label='Macro F1', color=pal_f1, edgecolor='black', alpha=0.9)
    ax.axhline(33.33, color='#E63946', linestyle='--', linewidth=1.8, label='Chance Level (33.3%)')
    ax.axhline(base_val, color='#6C7A89', linestyle=':', linewidth=2.0, label=f'Majority Baseline ({base_val:.1f}%)')

    ax.set_title(f'3-Class {task} Classification', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Majority\nBaseline', 'Logistic\nRegression', 'Linear\nSVM', 'RBF\nSVM', 'Random\nForest'], fontsize=9)
    ax.set_ylim(0, 75)
    ax.legend(loc='upper right', frameon=True, fontsize=9)

fig_bar.suptitle('Figure 12: SER Model Performance Comparison (Stratified 5-Fold CV)', fontsize=14, fontweight='bold', y=0.98)
fig_bar.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 7. Conclusions & Findings
1. **Valence Ambiguity:** Random Forest achieves $32.5\%$ accuracy, reflecting that acoustic features alone cannot easily distinguish positive from negative valence in spontaneous classroom speech.
2. **Arousal Predictability:** Linear SVM achieves $47.5\%$ in 5-fold CV and $55.0\%$ in LOOCV, significantly exceeding the $40.0\%$ baseline. Pitch and vocal energy reliably track physiological activation.
3. **Cross-Validation Necessity:** Large fold variances ($\pm 6.1\%$ to $\pm 18.7\%$) demonstrate that cross-validation is essential for reliable reporting on $N=40$.